In [1]:
 #*******************************************************************************************
 #
 #  File Name:  vacations.ipynb 
 #
 #  File Description:
 #      This interactive Python notebook, vacations.ipynb, is a Python script to 
 #      determine the ideal locations (city and hotel) for a vacation and displays
 #      information on a map.
 #      
 #
 #  Date            Description                             Programmer
 #  ----------      ------------------------------------    ------------------
 #  08/26/2023      Initial Development                     Nicholas J. George
 #
 #******************************************************************************************/

import citipyx

import logx
import pandasx

import warnings

import pandas as pd

from bokeh.util.warnings import BokehUserWarning


warnings.filterwarnings('ignore')

warnings.simplefilter(action = 'ignore', category = BokehUserWarning)


pd.options.mode.chained_assignment = None

In [2]:
CONSTANT_LOCAL_FILE_NAME = 'vacations.ipynb'

logx.set_log_mode(False)

logx.set_image_mode(False)


logx.begin_program('vacations')

In [3]:
citipyx.set_vacation_temp_range(70, 95)

citipyx.set_vacation_humid_range(35, 65)

citipyx.set_vacation_cloud_range(0, 10)

citipyx.set_vacation_wind_speed_range(0, 10)

# <br> **Section 1: Vacation Data Acquisition**

## **1.1: Data Import from CSV File**

In [4]:
city_weather_df \
    = pd.read_csv \
        (citipyx.config_dict['data']['datafile'],
         index_col = citipyx.config_dict['params']['index'])

logx.log_write_obj(city_weather_df)

## **1.2: Display City Weather Data Set**

In [5]:
pandasx.return_format_table(city_weather_df, 'Table: 1.2: City Weather Information')

city,latitude,longitude,temperature,humidity,cloudiness,wind_speed,country,date_time
oranjemund,-28.55,16.43,61.14,94,7,9.66,nan,2026-01-27 17:48:12
howard springs,-12.50,131.05,85.91,88,100,9.40,AU,2026-01-27 17:48:12
yellowknife,62.46,-114.35,-10.68,83,75,10.36,CA,2026-01-27 17:45:48
bethel,41.37,-73.41,12.04,76,0,4.61,US,2026-01-27 17:48:12
bredasdorp,-34.53,20.04,61.79,83,7,1.01,ZA,2026-01-27 17:48:12
adamstown,-25.07,-130.10,77.22,77,1,16.35,PN,2026-01-27 17:48:12
gaigeturi,33.46,126.32,35.65,51,75,9.22,KR,2026-01-27 17:48:12
talnakh,69.49,88.40,-25.01,95,88,7.58,RU,2026-01-27 17:48:13
luderitz,-26.65,15.16,64.62,73,3,16.62,nan,2026-01-27 17:48:13
avarua,-21.21,-159.78,80.65,89,100,19.57,CK,2026-01-27 17:48:13


## **1.3: Display City Weather Locations**

In [6]:
hover_cols_list \
    = ['city', 'latitude', 'longitude', 'temperature', 
       'humidity', 'cloudiness', 'wind_speed', 'country']

pandasx.display_df_hvplot \
    (city_weather_df,
     'Figure 1.3: City Weather Locations',
     'city', 'humidity', 'longitude', 'latitude',
     hover_columns_list = hover_cols_list)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [longitude,latitude]   (city,humidity,temperature,cloudiness,wind_speed,country)

# <br> **Section 2: Desired Weather Locations**

## **2.1: Establish Desired Weather Conditions for Vacation Locations**

In [7]:
vacations_df \
    = city_weather_df \
        .loc[(city_weather_df['temperature'] \
                >= citipyx.ranges_dict['temp'][0]) \
             & (city_weather_df['temperature'] \
                <= citipyx.ranges_dict['temp'][1]), :]

vacations_df \
    = vacations_df \
        .loc[(vacations_df['humidity'] \
                >= citipyx.ranges_dict['humid'][0]) \
             & (vacations_df['humidity'] \
                <= citipyx.ranges_dict['humid'][1]), :]

vacations_df \
    = vacations_df \
        .loc[(vacations_df['cloudiness'] \
                >= citipyx.ranges_dict['cloud'][0]) \
             & (vacations_df['cloudiness'] \
                <= citipyx.ranges_dict['cloud'][1]), :]

vacations_df \
    = vacations_df \
        .loc[(vacations_df['wind_speed'] \
                >= citipyx.ranges_dict['wind_speed'][0]) 
             & (vacations_df['wind_speed'] \
                <= citipyx.ranges_dict['wind_speed'][1]), :]

vacations_df.dropna(inplace = True)

vacations_df.reset_index(drop = True, inplace = True)

logx.log_write_obj(vacations_df)

## **2.2: Display Vacation Data Set**

In [8]:
pandasx.return_format_table(vacations_df, 'Table: 2.3: Vacation Locations')

city,latitude,longitude,temperature,humidity,cloudiness,wind_speed,country,date_time
kete krachi,7.79,-0.05,81.77,61,0,7.58,GH,2026-01-27 17:48:34
mochudi,-24.42,26.15,72.19,45,5,5.79,BW,2026-01-27 17:48:34
jeddah,21.52,39.22,76.86,65,0,0.00,SA,2026-01-27 17:47:23
port macquarie,-31.43,152.92,79.07,63,0,8.99,AU,2026-01-27 17:48:55
san carlos centro,-31.73,-61.09,80.31,54,1,7.25,AR,2026-01-27 17:49:26


## **2.3: Display Vacation Locations**

In [9]:
pandasx.display_df_hvplot \
    (vacations_df,
     'Figure 2.4: Vacation Locations',
     'city', 'humidity', 'longitude', 'latitude',
     hover_columns_list = hover_cols_list)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [longitude,latitude]   (city,humidity,temperature,cloudiness,wind_speed,country)

# <br> **Section 3: Hotel Locations**

## **3.1: Add Hotel Column to DataFrame**

In [10]:
hotels_df = vacations_df.copy()

hotels_df['hotel_name'] = pd.Series(dtype = 'str')

hotels_df.reset_index(drop = True, inplace = True)

logx.log_write_obj(hotels_df)

## **3.2: Find Hotel Locations**

In [11]:
updated_hotels_df \
    = citipyx.update_location_vacation_df \
        (hotels_df, 'hotel_name', 'accommodation.hotel', 10000)

logx.log_write_obj(updated_hotels_df)

STARTING HOTEL SEARCH...


Located the following hotel...Simon Hotel in kete krachi, GH


Located the following hotel...Rasesa Lodge in kete krachi, GH


Located the following hotel...فندق الاندلس in mochudi, BW


Located the following hotel...The Observatory in kete krachi, GH


Located the following hotel...Hotel Cenci in mochudi, BW


HOTEL SEARCH COMPLETE...




## **3.3: Display Hotel Data Set**

In [12]:
pandasx.return_format_table(updated_hotels_df, 'Table: 3.3: Hotel Locations')

city,latitude,longitude,temperature,humidity,cloudiness,wind_speed,country,date_time,hotel_name
kete krachi,7.79,-0.05,81.77,61,0,7.58,GH,2026-01-27 17:48:34,Simon Hotel
mochudi,-24.42,26.15,72.19,45,5,5.79,BW,2026-01-27 17:48:34,Rasesa Lodge
jeddah,21.52,39.22,76.86,65,0,0.00,SA,2026-01-27 17:47:23,فندق الاندلس
port macquarie,-31.43,152.92,79.07,63,0,8.99,AU,2026-01-27 17:48:55,The Observatory
san carlos centro,-31.73,-61.09,80.31,54,1,7.25,AR,2026-01-27 17:49:26,Hotel Cenci


## **3.4: Display Hotel Locations**

In [13]:
hover_cols_list \
    = ['city', 'latitude', 'longitude', 'humidity', 'hotel_name', 'country']

pandasx.display_df_hvplot \
    (updated_hotels_df,
     'Figure 3.4: Hotel Locations',
     'city', 'humidity', 'longitude', 'latitude',
     hover_columns_list = hover_cols_list)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [longitude,latitude]   (city,humidity,hotel_name,country)

# <br> **Section 4: Restaurant Locations**

## **4.1: Add Restaurant Column to DataFrame**

In [14]:
restaurant_df = updated_hotels_df.copy()

restaurant_df['restaurant_name'] = pd.Series(dtype = 'str')

restaurant_df.reset_index(drop = True, inplace = True)

logx.log_write_obj(restaurant_df)

## **4.2: Find Restaurant Locations**

In [15]:
updated_restaurant_df \
    = citipyx.update_location_vacation_df \
        (restaurant_df, 'restaurant_name', 'catering.restaurant', 10000)

logx.log_write_obj(updated_restaurant_df)

STARTING RESTAURANT SEARCH...


Located the following restaurant...Honchos in kete krachi, GH


Located the following restaurant...فول عباس in kete krachi, GH


Located the following restaurant...scampis in kete krachi, GH


RESTAURANT SEARCH COMPLETE...




## **4.3: Display Restaurant Data Set**

In [16]:
pandasx.return_format_table(updated_restaurant_df, 'Table: 4.3: Restaurant Locations')

city,latitude,longitude,temperature,humidity,cloudiness,wind_speed,country,date_time,hotel_name,restaurant_name
kete krachi,7.79,-0.05,81.77,61,0,7.58,GH,2026-01-27 17:48:34,Simon Hotel,Honchos
mochudi,-24.42,26.15,72.19,45,5,5.79,BW,2026-01-27 17:48:34,Rasesa Lodge,فول عباس
jeddah,21.52,39.22,76.86,65,0,0.00,SA,2026-01-27 17:47:23,فندق الاندلس,scampis
port macquarie,-31.43,152.92,79.07,63,0,8.99,AU,2026-01-27 17:48:55,The Observatory,nan


## **4.4: Display Restaurant Locations**

In [17]:
hover_cols_list \
    = ['city', 'latitude', 'longitude', 'humidity', 'hotel_name', 'restaurant_name', 'country']

pandasx.display_df_hvplot \
    (updated_restaurant_df,
     'Figure 4.4: Restaurant Locations',
     'city', 'humidity', 'longitude', 'latitude',
     hover_columns_list = hover_cols_list)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [longitude,latitude]   (city,humidity,hotel_name,restaurant_name,country)

# <br> **Section 5: Tourism Attraction Locations**

## **5.1: Add Tourism Attraction Column to DataFrame**

In [18]:
tourist_attraction_df = updated_restaurant_df.copy()

tourist_attraction_df['tourist_attraction'] = pd.Series(dtype = 'str')

tourist_attraction_df.reset_index(drop = True, inplace = True)

logx.log_write_obj(tourist_attraction_df)

## **5.2: Find Tourism Attraction Locations**

In [19]:
updated_tourist_attraction_df \
    = citipyx.update_location_vacation_df \
        (tourist_attraction_df, 'tourist_attraction', 'tourism.attraction', 10000)

logx.log_write_obj(updated_tourist_attraction_df)

STARTING TOURISM ATTRACTION SEARCH...


Located the following tourism attraction...Khuzam Palace Main Gate in kete krachi, GH


Located the following tourism attraction...Port Macquarie Observatory in mochudi, BW


TOURISM ATTRACTION SEARCH COMPLETE...




## **5.3: Display Tourism Attraction Data Set**

In [20]:
pandasx.return_format_table \
    (updated_tourist_attraction_df, 
     'Table: 5.3: Tourist Attraction Locations')

city,latitude,longitude,temperature,humidity,cloudiness,wind_speed,country,date_time,hotel_name,restaurant_name,tourist_attraction
jeddah,21.52,39.22,76.86,65,0,0.00,SA,2026-01-27 17:47:23,فندق الاندلس,scampis,Khuzam Palace Main Gate
port macquarie,-31.43,152.92,79.07,63,0,8.99,AU,2026-01-27 17:48:55,The Observatory,nan,Port Macquarie Observatory


## **5.4: Display Tourism Attraction Locations**

In [21]:
hover_cols_list \
    = ['city', 'latitude', 'longitude', 'humidity', 'hotel_name', 
       'restaurant_name', 'tourist_attraction', 'country']

pandasx.display_df_hvplot \
    (updated_tourist_attraction_df,
     'Figure 5.4: Tourist Attraction Locations',
     'city', 'humidity', 'longitude', 'latitude',
     hover_columns_list = hover_cols_list)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [longitude,latitude]   (city,humidity,hotel_name,restaurant_name,tourist_attraction,country)

In [22]:
# log_subroutine.end_program()